In [406]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 


In [407]:
df = pd.read_csv("qoute_dataset.csv")

In [408]:
df

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
...,...,...
3033,The past beats inside me like a second heart.,"John Banville,"
3034,"Damn, Claire. Warn a guy before you do a face-...","Rachel Caine,"
3035,"Can you be a girl for a few seconds?""""I'm alwa...","Veronica Roth,"
3036,That's what fiction is for. It's for getting a...,Tim O'Brien


In [409]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [410]:
df.shape

(3038, 2)

In [411]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [412]:
quotes = quotes.str.lower()

In [413]:
import string
translator = str.maketrans('','',string.punctuation) 
quotes = quotes.apply(lambda x:x.translate(translator)) 

In [414]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [415]:
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore

In [416]:
vocab_size = 10000

tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quotes) 

In [417]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10] 

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [418]:
sequence = tokenizer.texts_to_sequences(quotes)

In [419]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [420]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [421]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [422]:
len(X)

85271

In [423]:
len(y)

85271

In [424]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [425]:
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
X_padded = pad_sequences(X, maxlen = max_len, padding='pre')

In [426]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [427]:
y = np.array(y)


In [428]:
X_padded.shape

(85271, 745)

In [429]:
from tensorflow.keras.utils import to_categorical  # type: ignore
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [430]:
y.shape

(85271,)

In [431]:
y_one_hot.shape

(85271, 10000)

In [432]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense # type: ignore

In [433]:
embedding_dim = 50
rnn_units = 128

In [434]:
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim = vocab_size,output_dim = embedding_dim,
    input_length = max_len)
    )

rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units = vocab_size,activation = 'softmax'))

c:\Users\Nemo96\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [435]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [436]:
rnn_model.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [437]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim = vocab_size,output_dim = embedding_dim,
    input_length = max_len)
)
lstm_model.add(LSTM(units = rnn_units))
lstm_model.add(Dense(units = vocab_size,activation = 'softmax'))

In [438]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [439]:
lstm_model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [440]:
# history_rnn = rnn_model.fit(
#     X_padded, y_one_hot,
#     epochs = epochs,
#     batch_size = batch_size,
#     validation_split = 0.1
# )

In [441]:
# epochs = 100
# batch_size = 128

# history_lstm = lstm_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs = epochs,
#     batch_size = batch_size,
#     validation_split = 0.1
# )

In [442]:
lstm_model.save("lstm_model.h5")

In [443]:
from tensorflow.keras.models import load_model # type: ignore

lstm_model = load_model("lstm_model.h5")

In [444]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [445]:
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore

In [446]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [447]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

plum


In [448]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [449]:
seed = "i you can"
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

i you can date across date sensitive conviction struggled struggled inheritance inheritance mantle


In [450]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [451]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)